# Multi-Cancer Dataset Expansion
**Genomic-RawSeq-Analyzer — Semester 2**

Downloads and preprocesses two new WXS cohorts from NCBI SRA to test
the pipeline's generalizability across cancer types:

| Cohort | Type | GEO | Samples |
|--------|------|-----|---------|
| **BRCA** | Breast Invasive Carcinoma | GSE48215 | 25 Tumor + 25 Normal |
| **LUAD** | Lung Adenocarcinoma | GSE40419 | 17 Tumor + 13 Normal |

After download, evaluates the **Semester 1 CNN** on each new cohort **zero-shot**
(no retraining) to measure cross-cancer transfer performance.

**Steps:**
1. Install `sra-tools` and download FASTQ files via `fasterq-dump`
2. Preprocess using existing `data_loader.py`
3. Check class balance
4. Run zero-shot CNN evaluation (read-level + patient-level AUC)

**Outputs saved to Google Drive:**
- `results/multi_cancer/brca/` — batch .npz files
- `results/multi_cancer/luad/` — batch .npz files
- `results/multi_cancer/brca/zero_shot_eval/`
- `results/multi_cancer/luad/zero_shot_eval/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/492'
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install SRA tools for FASTQ download
!pip install -q tensorflow biopython scikit-learn matplotlib
!apt-get install -q -y sra-toolkit
!which fasterq-dump && fasterq-dump --version || echo 'fasterq-dump not found — using prefetch fallback'

In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
from data_loader import DataLoader
from multi_cancer_loader import (
    COHORT_METADATA,
    assign_labels_from_sra_table,
    class_balance_report,
    zero_shot_eval,
)
print('Imports OK')

## Configuration
Set `CANCER_TYPE` to `'brca'` or `'luad'`. Run this notebook twice to process both.

In [ ]:
# ── Choose cohort ──────────────────────────────────────────────────
CANCER_TYPE       = 'brca'     # 'brca' or 'luad'
MAX_READS         = 50_000     # reads per patient (matches Semester 1)
CNN_MODEL_PATH    = 'results/cnn_baseline.keras'

meta       = COHORT_METADATA[CANCER_TYPE]
OUTPUT_DIR = f'results/multi_cancer/{CANCER_TYPE}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Cohort  : {meta["cancer_label"]}')
print(f'GEO     : {meta["geo_accession"]}   SRA: {meta["sra_project"]}')
print(f'Expected: {meta["n_tumor"]} tumor + {meta["n_normal"]} normal')
print(f'Output  : {OUTPUT_DIR}')

## Step 1 — Get SRA Run List
**Option A (recommended):** Download the SraRunTable from NCBI Run Selector and upload it.  
**Option B:** Use the small hardcoded subset in `multi_cancer_loader.py` (8 samples).

For Option A:
1. Go to: https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SRP028580 (BRCA) or https://www.ncbi.nlm.nih.gov/Traces/study/?acc=SRP013469 (LUAD)
2. Click **Metadata** → download `SraRunTable.txt`
3. Upload to this Colab session

In [ ]:
SRA_TABLE_PATH = None   # set to '/content/SraRunTable.txt' if you uploaded it

if SRA_TABLE_PATH and os.path.exists(SRA_TABLE_PATH):
    run_df = assign_labels_from_sra_table(SRA_TABLE_PATH)
    # Filter to WXS only
    if 'LibraryStrategy' in run_df.columns:
        run_df = run_df[run_df['LibraryStrategy'] == 'WXS']
    print(f'Using SraRunTable: {len(run_df)} runs')
else:
    # Fallback: hardcoded subset
    run_df = pd.DataFrame([
        {'Run': acc, 'Label': lbl}
        for acc, lbl in meta['runs'].items()
    ])
    print(f'Using hardcoded subset: {len(run_df)} runs')
    print('TIP: Upload SraRunTable.txt for the full cohort.')

print(f'\nRuns to download: {len(run_df)}')
print(f'  Tumor  : {(run_df["Label"]==1).sum()}')
print(f'  Normal : {(run_df["Label"]==0).sum()}')
display(run_df.head(10))

## Step 2 — Download FASTQ Files via fasterq-dump
Each sample downloads ~500 MB. With 8 samples this takes ~10-15 min on Colab.

In [ ]:
FASTQ_DIR = f'/content/fastq_{CANCER_TYPE}'
os.makedirs(FASTQ_DIR, exist_ok=True)

print(f'Downloading {len(run_df)} FASTQ files to {FASTQ_DIR}...')
for i, row in run_df.iterrows():
    acc = row['Run']
    outfile = os.path.join(FASTQ_DIR, f'{acc}_1.fastq')
    if os.path.exists(outfile):
        print(f'  {acc} already downloaded, skipping.')
        continue
    print(f'  [{i+1}/{len(run_df)}] Downloading {acc}...')
    ret = os.system(
        f'fasterq-dump {acc} --outdir {FASTQ_DIR} '
        f'--split-files --threads 4 --progress '
        f'-X {MAX_READS} 2>&1'
    )
    if ret != 0:
        print(f'    WARNING: fasterq-dump exited with code {ret} for {acc}')

fastq_files = [f for f in os.listdir(FASTQ_DIR) if f.endswith('.fastq')]
print(f'\nDownloaded {len(fastq_files)} FASTQ files.')

## Step 3 — Preprocess: Integer-Encode & Save Batches

In [ ]:
print(f'Processing {len(run_df)} samples into integer-encoded batches...')
loader = DataLoader(output_dir=OUTPUT_DIR, max_reads=MAX_READS)
loader.process_run_list(run_df, fastq_dir=FASTQ_DIR)
print(f'Batches saved to: {OUTPUT_DIR}')

## Step 4 — Class Balance Report

In [ ]:
class_balance_report(OUTPUT_DIR, cancer_label=meta['cancer_label'])

## Step 5 — Zero-Shot CNN Evaluation
Evaluates the Semester 1 CNN (trained on WXS breast cancer) on the new cohort
**without any retraining** to measure cross-cancer transfer performance.

In [ ]:
eval_dir = f'{OUTPUT_DIR}/zero_shot_eval'
zero_shot_eval(
    model_path=CNN_MODEL_PATH,
    batch_dir=OUTPUT_DIR,
    cancer_label=meta['cancer_label'],
    save_dir=eval_dir,
)
print(f'\nZero-shot evaluation plots saved to: {eval_dir}')

## Summary

| Metric | WXS Breast (Sem. 1) | New Cohort (zero-shot) |
|--------|--------------------|-----------------------|
| Read-level AUC | 0.6157 | *see output above* |
| Patient-level AUC | 0.9156 | *see output above* |

**Interpretation:**
- If zero-shot AUC > 0.55: the CNN learned transferable somatic mutation features
- If zero-shot AUC ≈ 0.50: the signal is cancer-type specific → fine-tuning needed
- Patient-level crowd-voting will still amplify any consistent signal